# Exploration - L2 Orthogonality of Repeated Tent Function Powers

In [144]:
from nonlinear_approximator.activations import tent
from nonlinear_approximator.params import TentParams
from functools import partial
import matplotlib.pyplot as plt
import numpy as np
import tqdm
import scipy.integrate as integrate

In [116]:
params = TentParams(mu=1.99)

In [162]:
# return a function defined as the n^th power of tent params applied as input. 
def gen_tent_func(params, n, cache: dict[int, callable]):
    if n == 0:
        raise RuntimeError("n >= 1")
    if n == 1:
        return partial(tent, params=params)
    else: 
        if n in cache.keys():
            return cache[n]
        else:
            nm1 = gen_tent_func(params, n-1, cache)
            out = lambda x: tent(x=nm1(x), params=TentParams(mu=1.99))
            cache[n] = out
            return out
        
def l2_norm(f1, f2, domain):
    dx = domain[1]-domain[0]
    return np.trapezoid(
        y = f1(domain) * f2(domain), 
        x=domain,
        dx = dx,
    )
    

In [163]:
one = gen_tent_func(params, 1, {})
two = gen_tent_func(params, 2, {})
three = gen_tent_func(params, 3, {})
xs = np.linspace(0, 1, 1000)
plt.figure("gen")
plt.clf()
plt.plot(xs, one(xs), marker='x', c='r')
plt.plot(xs, tent(xs, params), c='r')
plt.plot(xs, two(xs), marker='x', c='g')
plt.plot(xs, three(xs), marker='x', c='g')

plt.plot(xs, tent(
    tent(xs, params), params
), c='g')
# plt.plot(xs(tent(xs, params)))
plt.ion()
plt.show()

In [183]:
cache = {}
powers = []
num_pows = 100
for j in range(1, num_pows + 1):
    powers.append(
        gen_tent_func(params, j, cache)
    )

In [184]:
dx  = 1e-2
xs = np.linspace(0, 1, int(1/dx))
plt.figure("gen2")
plt.clf()
for power in powers:
    plt.plot(xs, power(xs))
plt.show()

In [185]:
inner_prods = np.zeros((num_pows, num_pows))
for i, pow_row in tqdm.tqdm(enumerate(powers), total=len(powers)):
    for j, pow_col in tqdm.tqdm(enumerate(powers[i:]), total=len(powers[i:]), leave=False):
        if j == 0: 
            continue
        else:
            norm = l2_norm(pow_row, pow_col, xs)
            inner_prods[i,i + j] = norm
            inner_prods[i+j, i] = norm


100%|██████████| 100/100 [00:02<00:00, 36.34it/s]


In [186]:

plt.figure("comp")
plt.clf()
plt.imshow(inner_prods, norm='symlog', vmin=-1, vmax=1)
plt.colorbar()
plt.show()
